In [96]:
import pandas as pd
import glob
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
import csv
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.layers import Input, Embedding, GlobalAveragePooling1D, Dense, Dropout, Bidirectional, LSTM
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [97]:
all_files = glob.glob("*.csv")  # atau kasih path spesifik: "data/*.csv"

df = pd.concat([
    pd.read_csv(f, engine='python', quoting=csv.QUOTE_MINIMAL, on_bad_lines='skip')
    for f in all_files
], ignore_index=True)

print(f"Total rows: {len(df)}")
print(f"Dari {len(all_files)} file: {all_files}")

df.info()
df.head()

Total rows: 10700
Dari 8 file: ['linkedin_jobs_20260416_223428_datscien.csv', 'linkedin_jobs_20260418_215155_webdev.csv', 'linkedin_jobs_20260426_002316.csv', 'jobstreet_cleaned terbaru.csv', 'linkedin_jobs_20260430_001046.csv', 'linkedin_jobs_20260505_063507.csv', 'linkedin_jobs_20260421_215330_softdev.csv', 'jobstreet_jobs_20260421_164240.csv']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10700 entries, 0 to 10699
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                10498 non-null  float64
 1   title             10498 non-null  object 
 2   company           10493 non-null  object 
 3   location          10434 non-null  object 
 4   job_description   10671 non-null  object 
 5   job_url           10498 non-null  object 
 6   search_role       10700 non-null  object 
 7   search_location   10700 non-null  object 
 8   extracted_skills  10700 non-null  object 
 9   skills_count      10

,id,title,company,location,job_description,job_url,search_role,search_location,extracted_skills,skills_count,scraped_at,salary
0,4.354913e+09,Data Analyst,ParagonCorp,"Jakarta, Indonesia (On-site)",About the job\n\nJob Description\n\nDeliver in...,https://www.linkedin.com/jobs/view/4354913165/...,Data Scientist,Indonesia,"['python', 'sql', 'tableau', 'power bi', 'r', ...",6,2026-04-16T21:04:46.182549,NaN
1,4.395935e+09,Intern,Schoters,Jakarta Metropolitan Area (Hybrid),About the job\n\nLooking for a chance to kicks...,https://www.linkedin.com/jobs/view/4395934526/...,Data Scientist,Indonesia,"['git', 'r']",2,2026-04-16T21:04:50.857008,NaN
2,4.401866e+09,Data Scientist,Artajasa Pembayaran Elektronis,"Kota Tangerang Selatan, Banten, Indonesia (On-...",About the job\n\nPT Artajasa Pembayaran Elektr...,https://www.linkedin.com/jobs/view/4401866027/...,Data Scientist,Indonesia,"['python', 'pandas', 'sql', 'numpy', 'scikit-l...",6,2026-04-16T21:04:55.554607,NaN
3,4.402503e+09,Data Scientist (Remote),Jobs Ai,APAC (Remote),About the job\n\nRole: Data Scientist (Remote)...,https://www.linkedin.com/jobs/view/4402503224/...,Data Scientist,Indonesia,"['python', 'sql', 'r']",3,2026-04-16T21:05:00.274453,NaN
4,4.392809e+09,Operation Administration Intern (3 Months),PT Lion Super Indo,"Jakarta, Indonesia (On-site)",About the job\n\nTanggung Jawab\n\nMembantu da...,https://www.linkedin.com/jobs/view/4392809495/...,Data Scientist,Indonesia,['r'],1,2026-04-16T21:05:04.958011,NaN


In [98]:
print(f"Jumlah unique role: {df['search_role'].nunique()}")
print()
print(df['search_role'].value_counts())

Jumlah unique role: 16

search_role
Data Engineer           1688
Data Scientist          1140
Software Developer      1000
Software Engineer       1000
Developer               1000
Software                1000
IT                      1000
Full stack Developer     944
Web Developer            831
Data Analyst             342
Business Analyst         294
Progammer                224
Ai Engineer               83
software                  64
Backend Developer         62
Business Development      28
Name: count, dtype: int64


In [99]:
df = df.drop_duplicates()
df = df.drop(columns='salary', errors='ignore')

print(f"total duplicated columns : {df.duplicated().sum()}")
print(f"total null column : {df.isnull().sum()}")

total duplicated columns : 0
total null column : id                  202
title               202
company             207
location            266
job_description      29
job_url             202
search_role           0
search_location       0
extracted_skills      0
skills_count          0
scraped_at            0
dtype: int64


In [100]:
df.head()

,id,title,company,location,job_description,job_url,search_role,search_location,extracted_skills,skills_count,scraped_at
0,4.354913e+09,Data Analyst,ParagonCorp,"Jakarta, Indonesia (On-site)",About the job\n\nJob Description\n\nDeliver in...,https://www.linkedin.com/jobs/view/4354913165/...,Data Scientist,Indonesia,"['python', 'sql', 'tableau', 'power bi', 'r', ...",6,2026-04-16T21:04:46.182549
1,4.395935e+09,Intern,Schoters,Jakarta Metropolitan Area (Hybrid),About the job\n\nLooking for a chance to kicks...,https://www.linkedin.com/jobs/view/4395934526/...,Data Scientist,Indonesia,"['git', 'r']",2,2026-04-16T21:04:50.857008
2,4.401866e+09,Data Scientist,Artajasa Pembayaran Elektronis,"Kota Tangerang Selatan, Banten, Indonesia (On-...",About the job\n\nPT Artajasa Pembayaran Elektr...,https://www.linkedin.com/jobs/view/4401866027/...,Data Scientist,Indonesia,"['python', 'pandas', 'sql', 'numpy', 'scikit-l...",6,2026-04-16T21:04:55.554607
3,4.402503e+09,Data Scientist (Remote),Jobs Ai,APAC (Remote),About the job\n\nRole: Data Scientist (Remote)...,https://www.linkedin.com/jobs/view/4402503224/...,Data Scientist,Indonesia,"['python', 'sql', 'r']",3,2026-04-16T21:05:00.274453
4,4.392809e+09,Operation Administration Intern (3 Months),PT Lion Super Indo,"Jakarta, Indonesia (On-site)",About the job\n\nTanggung Jawab\n\nMembantu da...,https://www.linkedin.com/jobs/view/4392809495/...,Data Scientist,Indonesia,['r'],1,2026-04-16T21:05:04.958011


In [101]:
import ast

def parse_skills(val):
    try:
        result = ast.literal_eval(val)
        return ' '.join(result) if isinstance(result, list) and len(result) > 0 else ''
    except:
        return ''

df['extracted_skills_clean'] = df['extracted_skills'].apply(parse_skills)

print(f"Data setelah filter: {len(df)} rows")
print(f"Skills masih kosong: {(df['extracted_skills_clean'] == '').sum()} rows")
print(df['search_role'].value_counts())

Data setelah filter: 10700 rows
Skills masih kosong: 39 rows
search_role
Data Engineer           1688
Data Scientist          1140
Software Developer      1000
Software Engineer       1000
Developer               1000
Software                1000
IT                      1000
Full stack Developer     944
Web Developer            831
Data Analyst             342
Business Analyst         294
Progammer                224
Ai Engineer               83
software                  64
Backend Developer         62
Business Development      28
Name: count, dtype: int64


In [102]:
min_samples = 100
role_counts = df['search_role'].value_counts()
valid_roles = role_counts[role_counts >= min_samples].index
df = df[df['search_role'].isin(valid_roles)].reset_index(drop=True)
print(df['search_role'].value_counts())

search_role
Data Engineer           1688
Data Scientist          1140
IT                      1000
Software Engineer       1000
Software Developer      1000
Software                1000
Developer               1000
Full stack Developer     944
Web Developer            831
Data Analyst             342
Business Analyst         294
Progammer                224
Name: count, dtype: int64


In [103]:
from sklearn.utils import resample

max_samples = 1500
dfs = []
for role, group in df.groupby('search_role'):
    if len(group) > max_samples:
        group = resample(group, n_samples=max_samples, random_state=42)
    dfs.append(group)

df = pd.concat(dfs).reset_index(drop=True)
print(df['search_role'].value_counts())

search_role
Data Engineer           1500
Data Scientist          1140
IT                      1000
Developer               1000
Software Engineer       1000
Software                1000
Software Developer      1000
Full stack Developer     944
Web Developer            831
Data Analyst             342
Business Analyst         294
Progammer                224
Name: count, dtype: int64


In [104]:
df['text_input'] = df['job_description'] + ' ' + df['extracted_skills_clean']

role_mapping = {
    'software':           'Software Engineer',
    'Software':           'Software Engineer',
    'Software Developer': 'Software Engineer',
    'Developer':          'Software Engineer',
    'Progammer':          'Software Engineer',
    'Full stack Developer': 'Fullstack Developer',
    'Web Developer':      'Fullstack Developer',
    'IT':                 'Software Engineer',
}


df['search_role'] = df['search_role'].replace(role_mapping)
print(df['search_role'].value_counts())

encoder = LabelEncoder()
df['label'] = encoder.fit_transform(df['search_role'])

num_classes = len(encoder.classes_)
print(f"Total role unik: {num_classes}")

print(f"Mapping kelas:\n{dict(zip(encoder.classes_, range(num_classes)))}")

df['text_input'] = df['text_input'].fillna('').astype(str)
df = df[df['text_input'].str.strip() != ''].reset_index(drop=True)

X_train, X_val, y_train, y_val = train_test_split(
    df['text_input'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

max_vocab_length = 20000
max_sequence_length = 512

vectorizer = TextVectorization(
    max_tokens=max_vocab_length,
    output_mode='int',
    output_sequence_length=max_sequence_length
)

vectorizer.adapt(X_train.to_numpy())

print("\nShape X_train:", X_train.shape)
print("Contoh kalimat pertama setelah di-vektorisasi (angka):")
print(vectorizer(X_train.iloc[0:1]).numpy())

search_role
Software Engineer      5224
Fullstack Developer    1775
Data Engineer          1500
Data Scientist         1140
Data Analyst            342
Business Analyst        294
Name: count, dtype: int64
Total role unik: 6
Mapping kelas:
{'Business Analyst': 0, 'Data Analyst': 1, 'Data Engineer': 2, 'Data Scientist': 3, 'Fullstack Developer': 4, 'Software Engineer': 5}

Shape X_train: (8196,)
Contoh kalimat pertama setelah di-vektorisasi (angka):
[[   21     4    18    22  2939  3603     3  4203  9206     3  9208  8410
   1459   619  2673 12735  1274    41  1098    13    19   594     8  1157
    365   620   251     7   309   563     3   191   414   200   112    35
    277  2695     2  1979     8  1309    35    25    42    15     8   195
     40    41     7  2122   788     9    25   570     2   258    35   114
     98    43     2    71   677   187  1255   252     2    23   841    82
   6237     2  2422    67  1172     2  2969   187  1255     7  4457   224
   1741  1045     2   855   9

In [105]:
print(type(X_train), X_train.dtype)
print(X_train.head(3))
print(X_train.astype(str).to_numpy().dtype)

<class 'pandas.core.series.Series'> object
4676    About the job\n\nWork Duration\n\n\n Monday to...
4190    About the job\n\nCompany Overview\n\nDidirikan...
9688    About the job\nThis job is sourced from a job ...
Name: text_input, dtype: object
object


In [106]:
import numpy as np

class CustomEarlyStopping(tf.keras.callbacks.Callback):
    def __init__(self, patience=3):
        super(CustomEarlyStopping, self).__init__()
        self.patience = patience
        self.best_weights = None
        self.best_loss = np.inf
        self.wait = 0

    def on_epoch_end(self, epoch, logs=None):
        current_loss = logs.get('val_loss')

        if current_loss < self.best_loss:
            self.best_loss = current_loss
            self.wait = 0
            self.best_weights = self.model.get_weights()
        else:
            self.wait += 1
            if self.wait >= self.patience:
                print(f"\n[INFO] val_loss tidak membaik selama {self.patience} epoch. Overfitting terdeteksi. Stop training!")
                self.model.stop_training = True
                print("[INFO] Mengembalikan bobot model ke epoch terbaik.")
                self.model.set_weights(self.best_weights)

In [107]:
vocab_size = len(vectorizer.get_vocabulary())

inputs = Input(shape=(1,), dtype=tf.string, name='text_input')
x = vectorizer(inputs)
x = Embedding(input_dim=vocab_size, output_dim=128)(x)
x = Bidirectional(LSTM(128, return_sequences=True))(x)
x = Bidirectional(LSTM(64, return_sequences=False))(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.2)(x)

outputs = Dense(num_classes, activation='softmax', name='role_output')(x)

model = Model(inputs=inputs, outputs=outputs, name="Job_Role_Classifier_LSTM")

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nMulai proses training...")

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))

history = model.fit(
    X_train.to_numpy(), y_train,
    validation_data=(X_val.to_numpy(), y_val),
    epochs=15,
    batch_size=64,
    class_weight=class_weight_dict,
    callbacks=[CustomEarlyStopping(patience=7)]
)



Mulai proses training...
Epoch 1/15
129/129 ━━━━━━━━━━━━━━━━━━━━ 18s 112ms/step - accuracy: 0.2374 - loss: 1.7598 - val_accuracy: 0.2654 - val_loss: 1.5918
Epoch 2/15
129/129 ━━━━━━━━━━━━━━━━━━━━ 14s 109ms/step - accuracy: 0.2837 - loss: 1.4682 - val_accuracy: 0.3059 - val_loss: 1.4278
Epoch 3/15
129/129 ━━━━━━━━━━━━━━━━━━━━ 14s 107ms/step - accuracy: 0.3097 - loss: 1.3407 - val_accuracy: 0.2317 - val_loss: 1.3897
Epoch 4/15
129/129 ━━━━━━━━━━━━━━━━━━━━ 14s 106ms/step - accuracy: 0.3564 - loss: 1.2193 - val_accuracy: 0.4098 - val_loss: 1.2572
Epoch 5/15
129/129 ━━━━━━━━━━━━━━━━━━━━ 14s 108ms/step - accuracy: 0.4209 - loss: 1.0433 - val_accuracy: 0.4000 - val_loss: 1.1369
Epoch 6/15
129/129 ━━━━━━━━━━━━━━━━━━━━ 14s 112ms/step - accuracy: 0.4344 - loss: 0.9672 - val_accuracy: 0.4220 - val_loss: 1.1380
Epoch 7/15
129/129 ━━━━━━━━━━━━━━━━━━━━ 14s 106ms/step - accuracy: 0.4571 - loss: 0.9199 - val_accuracy: 0.4049 - val_loss: 1.1298
Epoch 8/15
129/129 ━━━━━━━━━━━━━━━━━━━━ 14s 107ms/step - 

In [108]:
model_path = 'job_role_model.keras'
model.save(model_path)
print(f"[INFO] Model berhasil diekspor ke: {model_path}")

loaded_model = tf.keras.models.load_model(model_path)

def rekomendasi_role(teks_skill_user):
    input_tensor = tf.constant([teks_skill_user], dtype=tf.string)

    pred_probs = loaded_model.predict(input_tensor, verbose=0)

    pred_class_idx = np.argmax(pred_probs)

    predicted_role = encoder.inverse_transform([pred_class_idx])[0]

    return predicted_role

# 3. Test fungsi inference
skill_input = "saya punya pengalaman bikin antarmuka web pakai reactjs, next.js, tailwind, dan biasa pakai linux ubuntu"
hasil_prediksi = rekomendasi_role(skill_input)

print("\n--- Hasil Inference ---")
print(f"Skill User : {skill_input}")
print(f"Cocok untuk: {hasil_prediksi}")

[INFO] Model berhasil diekspor ke: job_role_model.keras



--- Hasil Inference ---
Skill User : saya punya pengalaman bikin antarmuka web pakai reactjs, next.js, tailwind, dan biasa pakai linux ubuntu
Cocok untuk: Data Engineer
